# Exoplanet Detection Pipeline — BAH 2026, Problem Statement 7
### *AI-enabled Detection of Exoplanets from Noisy Astronomical Light Curves*

This notebook is the cell-by-cell version of the pipeline — run each cell
top to bottom and you'll see every result (tables, metrics, plots) appear
live, exactly like the script version but interactive.

**Before you start:** put `koi_table.csv` (the NASA Exoplanet Archive KOI
cumulative table) in the **same folder as this notebook**. If you're using
Google Colab instead, run the upload cell in Section 0 to upload it.

**What this notebook actually does, in one line:** it trains a classifier
that separates real confirmed exoplanets from false-positive signals
(eclipsing binaries, instrument noise) using physically meaningful transit
parameters from the KOI table — while explicitly avoiding a data-leakage
trap hidden in the same file (explained in Section 2).


## Section 0 — Setup

If this is a fresh Python environment, run this once in your terminal
**before** opening the notebook (not inside a cell — auto-installing from
inside a notebook is unreliable across different machines):

```
pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib
```

(On some systems, e.g. newer Ubuntu/Debian, you may need
`pip install --break-system-packages ...` or to use a virtual environment.)

The cell below just checks whether everything is already installed.


In [ ]:
# Checks whether everything you need is installed - does NOT try to
# install anything automatically (auto-install inside a notebook is
# unreliable across different machines/Python setups). If something is
# missing, copy the printed pip command into your terminal and run it
# there, then re-run this cell.
import importlib

IMPORT_TO_PIP = {"sklearn": "scikit-learn"}
required = ["pandas", "numpy", "sklearn", "xgboost", "matplotlib", "seaborn", "joblib"]

missing = []
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)

if missing:
    pip_cmd = "pip install " + " ".join(IMPORT_TO_PIP.get(m, m) for m in missing)
    print("Missing packages:", missing)
    print("Run this in your terminal, then re-run this cell:")
    print(" ", pip_cmd)
else:
    print("All required packages are already installed - you're good to go.")


In [ ]:
# OPTIONAL - only run this cell if you're on Google Colab and need to
# upload koi_table.csv manually (skip this if the file is already in
# the same folder as this notebook / you're running locally).

# from google.colab import files
# uploaded = files.upload()   # choose koi_table.csv when prompted


In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import os
os.makedirs("outputs", exist_ok=True)

%matplotlib inline
plt.rcParams["figure.dpi"] = 100


## Section 1 — Load and explore the raw data

Quick reminder of what this file actually is: NASA's **KOI cumulative
table** — one row per *Kepler Object of Interest*, with parameters already
derived from fitting a transit model to that star's light curve. It is
**not** a raw flux time series.


In [ ]:
DATA_PATH = "koi_table.csv"   # edit this path if your file is elsewhere

df_raw = pd.read_csv(DATA_PATH, comment="#")
print(f"Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()


In [ ]:
TARGET = "koi_disposition"
df_raw = df_raw.dropna(subset=[TARGET]).copy()

print(df_raw[TARGET].value_counts())
df_raw[TARGET].value_counts().plot(kind="bar", color=["#c53030", "#2b6cb0", "#d69e2e"])
plt.title("Class distribution")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()


## Section 2 — The data-leakage trap (read this before building features)

The table contains a few columns that are **NASA's own vetting decision**,
not independent physical measurements:

- `koi_score` — the Exoplanet Archive's own disposition confidence score
- `koi_pdisposition` — the Kepler pipeline's own disposition
- `koi_fpflag_nt`, `koi_fpflag_ss`, `koi_fpflag_co`, `koi_fpflag_ec` — flags
  that directly feed into how `koi_disposition` was assigned

If we train on these, the model gets a huge accuracy boost by reading the
answer key, not by learning real transit-signal physics. We exclude them
from the real model below, and we'll deliberately re-include them later
in Section 8 just to **prove** how much they inflate the score — that
ablation is worth a slide in your PPT.


In [ ]:
LEAKY_COLS = [
    "koi_score", "koi_pdisposition",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec",
    "koi_disp_prov", "koi_comment", "koi_vet_stat", "koi_vet_date",
]

ID_COLS = [
    "rowid", "kepid", "kepoi_name", "kepler_name",
    "koi_datalink_dvr", "koi_datalink_dvs", "koi_tce_delivname",
    "koi_parm_prov", "koi_sparprov", "koi_limbdark_mod", "koi_trans_mod",
    "koi_fittype", "koi_tce_plnt_num", "koi_quarters", "ra", "dec",
]

# The physically meaningful, leakage-free features we DO use:
FEATURE_COLS = [
    # transit signal shape
    "koi_period", "koi_duration", "koi_depth", "koi_impact", "koi_ror",
    "koi_dor", "koi_prad", "koi_teq", "koi_insol", "koi_srho",
    # signal detection statistics
    "koi_model_snr", "koi_max_sngle_ev", "koi_max_mult_ev",
    "koi_num_transits", "koi_count", "koi_bin_oedp_sig",
    # host star properties
    "koi_steff", "koi_slogg", "koi_smet", "koi_srad", "koi_smass", "koi_kepmag",
    # REAL centroid-shift measurements (not flags!) - this is what powers
    # a genuine "reject blended binaries" vetting check
    "koi_dicco_msky", "koi_dikco_msky", "koi_fwm_stat_sig",
]

FEATURE_COLS = [c for c in FEATURE_COLS if c in df_raw.columns]
print(f"Using {len(FEATURE_COLS)} leakage-free engineered features:")
print(FEATURE_COLS)


In [ ]:
work = df_raw[FEATURE_COLS + [TARGET]].copy()

# Treat non-physical values (<=0 where physically impossible) as missing
positive_only = ["koi_period", "koi_duration", "koi_depth", "koi_prad",
                  "koi_model_snr", "koi_num_transits", "koi_srad", "koi_smass"]
for col in positive_only:
    if col in work.columns:
        work.loc[work[col] <= 0, col] = np.nan

# Robust clipping at the 0.5/99.5 percentile (handles extreme outliers
# without throwing away whole rows - the tabular equivalent of sigma-clipping)
for col in FEATURE_COLS:
    lo, hi = work[col].quantile([0.005, 0.995])
    work[col] = work[col].clip(lo, hi)

# Log-transform heavily skewed columns (periods/depths span orders of magnitude)
LOG_COLS = ["koi_period", "koi_depth", "koi_insol", "koi_model_snr",
            "koi_max_sngle_ev", "koi_max_mult_ev", "koi_dor"]
for col in LOG_COLS:
    if col in work.columns:
        work[col] = np.log10(work[col].clip(lower=1e-6))
        work.rename(columns={col: f"log_{col}"}, inplace=True)

FEATURE_COLS = [c if c not in LOG_COLS else f"log_{c}" for c in FEATURE_COLS]

missing_report = work[FEATURE_COLS].isna().mean().sort_values(ascending=False)
print("Missing value fraction per feature (imputed later, no leakage across folds):")
missing_report[missing_report > 0]


## Section 3 — Train / test split

Stratified 80/20 split so the class balance is preserved in both sets.
Imputation is fit **only on the training set** and applied to the test
set — this avoids any leakage across the split.


In [ ]:
le = LabelEncoder()
X = work[FEATURE_COLS]
y = le.fit_transform(work[TARGET])
CLASS_NAMES = list(le.classes_)
print("Classes:", CLASS_NAMES)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)


## Section 4 — Train the models

Two models, trained for comparison:
- **Random Forest** — fast, interpretable baseline
- **XGBoost** — the "hero" model we'll keep and evaluate in depth


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=2, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train_imp, y_train)
rf_pred = rf.predict(X_test_imp)
rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average="macro")
print(f"[Random Forest]  accuracy={rf_acc:.4f}  macro-F1={rf_f1:.4f}")


In [ ]:
xgb = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    objective="multi:softprob", num_class=len(CLASS_NAMES),
    eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1
)
xgb.fit(X_train_imp, y_train)
xgb_pred = xgb.predict(X_test_imp)
xgb_proba = xgb.predict_proba(X_test_imp)

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred, average="macro")
print(f"[XGBoost]         accuracy={xgb_acc:.4f}  macro-F1={xgb_f1:.4f}")

report = classification_report(y_test, xgb_pred, target_names=CLASS_NAMES, output_dict=True)
print()
print(classification_report(y_test, xgb_pred, target_names=CLASS_NAMES))


## Section 5 — Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("XGBoost Confusion Matrix (leakage-free model)")
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png", dpi=150)
plt.show()


## Section 6 — Feature importance
What is the model actually using to make its decision?

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values()
plt.figure(figsize=(7, 7))
importances.tail(15).plot(kind="barh", color="#2b6cb0")
plt.title("Top 15 Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png", dpi=150)
plt.show()


## Section 7 — ROC / Precision-Recall curves

Headline binary metric for your PPT: collapsing CANDIDATE + CONFIRMED into
"planet-like" vs FALSE POSITIVE.


In [ ]:
fp_idx = CLASS_NAMES.index("FALSE POSITIVE")
y_test_binary = (y_test == fp_idx).astype(int)
proba_fp = xgb_proba[:, fp_idx]

roc_auc = roc_auc_score(y_test_binary, proba_fp)
pr_auc = average_precision_score(y_test_binary, proba_fp)
print(f"ROC-AUC = {roc_auc:.4f}   PR-AUC = {pr_auc:.4f}")

fpr, tpr, _ = roc_curve(y_test_binary, proba_fp)
prec, rec, _ = precision_recall_curve(y_test_binary, proba_fp)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, color="#2b6cb0", label=f"AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve (FALSE POSITIVE detection)"); axes[0].legend()

axes[1].plot(rec, prec, color="#c05621", label=f"AP = {pr_auc:.3f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()
plt.tight_layout()
plt.savefig("outputs/roc_pr_curves.png", dpi=150)
plt.show()


## Section 8 — Leakage ablation (the slide that shows you did this properly)

Same model, same split — but this time we deliberately let it cheat by
including `koi_score` and the `koi_fpflag_*` columns, to show numerically
how much the score inflates.


In [ ]:
leaky_extra_cols = ["koi_score", "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]

X_leaky = pd.concat([work[FEATURE_COLS], df_raw.loc[work.index, leaky_extra_cols]], axis=1)
X_leaky_train, X_leaky_test = X_leaky.loc[X_train.index], X_leaky.loc[X_test.index]

imputer_leak = SimpleImputer(strategy="median")
X_leaky_train_imp = imputer_leak.fit_transform(X_leaky_train)
X_leaky_test_imp = imputer_leak.transform(X_leaky_test)

xgb_leaky = XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    objective="multi:softprob", num_class=len(CLASS_NAMES),
    eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1
)
xgb_leaky.fit(X_leaky_train_imp, y_train)
leaky_pred = xgb_leaky.predict(X_leaky_test_imp)
leaky_acc = accuracy_score(y_test, leaky_pred)
leaky_f1 = f1_score(y_test, leaky_pred, average="macro")

print(f"Clean model        accuracy={xgb_acc:.4f}  macro-F1={xgb_f1:.4f}")
print(f"With leakage cols  accuracy={leaky_acc:.4f}  macro-F1={leaky_f1:.4f}")
print("\n^ The leaky version is reading NASA's own answer-key columns, not")
print("  detecting anything from real signal physics - exclude these in your real model.")

fig, ax = plt.subplots(figsize=(6, 4.5))
bars = ax.bar(["Clean model\n(no leakage)", "With koi_score\n+ fpflags (leakage)"],
              [xgb_acc, leaky_acc], color=["#2b6cb0", "#c53030"])
for bar, val in zip(bars, [xgb_acc, leaky_acc]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}", ha="center", fontweight="bold")
ax.set_ylim(0, 1.05); ax.set_ylabel("Test Accuracy")
ax.set_title("Why we excluded koi_score / fpflag_* columns")
plt.tight_layout()
plt.savefig("outputs/leakage_ablation.png", dpi=150)
plt.show()


## Section 9 — Illustrative phase-folded transit plot

**Important:** the KOI table has no raw flux column, so this plot
*simulates* a light curve using a real confirmed planet's (Kepler-227 b)
actual fitted period/duration/depth, plus injected noise — it's for
visual explanation only. Label it as a simulation if you put it in your
PPT. For a real measured curve, see Section 11.


In [ ]:
rng = np.random.default_rng(7)
row = df_raw[df_raw["kepoi_name"] == "K00752.01"].iloc[0]   # Kepler-227 b

period = row["koi_period"]
duration = row["koi_duration"] / 24
depth_ppm = row["koi_depth"]
depth_frac = depth_ppm / 1e6

cadence_min = 30 / 60 / 24
t = np.arange(0, period * 4, cadence_min)
flux = np.ones_like(t)
noise_sigma = max(depth_frac * 0.35, 0.0003)
flux += rng.normal(0, noise_sigma, size=t.shape)

phase = t % period
half_dur = duration / 2
in_transit = (phase < half_dur) | (phase > period - half_dur)
flux[in_transit] -= depth_frac
flux += 0.0006 * np.sin(2 * np.pi * t / (period * 2.7))   # slow stellar variability

window = int(period * 0.5 / cadence_min)
trend = pd.Series(flux).rolling(window, center=True, min_periods=1).median()
flat_flux = flux - trend.values + 1.0

fold_phase = (t % period) / period
fold_phase[fold_phase > 0.5] -= 1.0
order = np.argsort(fold_phase)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].scatter(t, flux, s=4, color="#a0aec0", alpha=0.6, label="Raw (simulated)")
axes[0].plot(t, trend, color="#c53030", lw=1.5, label="Detrending baseline")
axes[0].set_xlabel("Time [days]"); axes[0].set_ylabel("Relative flux")
axes[0].set_title(f"{row['kepler_name']} - simulated raw light curve")
axes[0].legend(loc="lower right", fontsize=8)

axes[1].scatter(fold_phase[order] * period * 24, flat_flux[order], s=5, color="#4a5568", alpha=0.5, label="Flattened flux")
axes[1].axvspan(-row["koi_duration"]/2, row["koi_duration"]/2, color="#e53e3e", alpha=0.15, label="Detected transit window")
axes[1].set_xlim(-row["koi_duration"]*3, row["koi_duration"]*3)
axes[1].set_xlabel("Hours from mid-transit"); axes[1].set_ylabel("Flattened relative flux")
axes[1].set_title(f"Phase-folded @ P={period:.3f} d | depth={depth_ppm:.0f} ppm")
axes[1].legend(loc="lower right", fontsize=8)

fig.suptitle(f"ILLUSTRATIVE SIMULATION based on {row['kepler_name']}'s real fitted parameters", fontsize=9, color="#718096")
plt.tight_layout()
plt.savefig("outputs/simulated_transit_example.png", dpi=150)
plt.show()


## Section 10 — Save the trained model

In [ ]:
joblib.dump(
    {"model": xgb, "imputer": imputer, "label_encoder": le, "feature_columns": FEATURE_COLS},
    "outputs/model_xgb.pkl"
)

summary = {
    "dataset_size": int(len(work)),
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "classes": CLASS_NAMES,
    "random_forest": {"accuracy": rf_acc, "macro_f1": rf_f1},
    "xgboost_clean": {"accuracy": xgb_acc, "macro_f1": xgb_f1, "per_class_report": report},
    "binary_planet_vs_fp": {"roc_auc": roc_auc, "pr_auc": pr_auc},
    "leakage_ablation": {"accuracy_with_leakage": leaky_acc, "accuracy_without_leakage": xgb_acc},
    "top_10_features": importances.tail(10).iloc[::-1].to_dict(),
}
with open("outputs/metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved model -> outputs/model_xgb.pkl")
print("Saved metrics -> outputs/metrics_summary.json")


## Section 11 — Final output: classify a new candidate + plain-English vetting notes

This is the function you'd wire up to a demo UI, or just call live on
stage with a few example candidates.


In [ ]:
def predict_candidate(feature_dict: dict) -> dict:
    LOG_RAW_NAMES = {
        "log_koi_period": "koi_period", "log_koi_depth": "koi_depth",
        "log_koi_insol": "koi_insol", "log_koi_model_snr": "koi_model_snr",
        "log_koi_max_sngle_ev": "koi_max_sngle_ev",
        "log_koi_max_mult_ev": "koi_max_mult_ev", "log_koi_dor": "koi_dor",
    }
    row = {}
    for col in FEATURE_COLS:
        if col in LOG_RAW_NAMES:
            raw_val = feature_dict.get(LOG_RAW_NAMES[col], np.nan)
            row[col] = np.log10(raw_val) if raw_val and raw_val > 0 else np.nan
        else:
            row[col] = feature_dict.get(col, np.nan)

    X_new = pd.DataFrame([row])[FEATURE_COLS]
    X_new_imp = imputer.transform(X_new)
    proba = xgb.predict_proba(X_new_imp)[0]
    pred_class = le.classes_[np.argmax(proba)]

    result = {
        "prediction": pred_class,
        "confidence": round(float(np.max(proba)), 4),
        "class_probabilities": {c: round(float(p), 4) for c, p in zip(le.classes_, proba)},
    }

    notes = []
    oedp = feature_dict.get("koi_bin_oedp_sig")
    if oedp is not None and oedp < 0.3:
        notes.append("Low odd/even transit depth significance -> possible eclipsing binary.")
    centroid = feature_dict.get("koi_dicco_msky") or feature_dict.get("koi_dikco_msky")
    if centroid is not None and centroid > 3:
        notes.append(f"Centroid shift of {centroid:.2f} arcsec -> signal may come from a nearby contaminating star.")
    if not notes:
        notes.append("No red flags from odd/even depth or centroid-shift checks.")
    result["vetting_notes"] = notes
    return result


In [ ]:
examples = {
    "Likely real planet (Kepler-227 b-like)": {
        "koi_period": 9.49, "koi_duration": 2.96, "koi_depth": 615.8,
        "koi_impact": 0.146, "koi_ror": 0.022, "koi_dor": 23.0,
        "koi_prad": 2.26, "koi_teq": 793, "koi_insol": 93.6, "koi_srho": 3.2,
        "koi_model_snr": 35.8, "koi_max_sngle_ev": 12.1, "koi_max_mult_ev": 30.5,
        "koi_num_transits": 142, "koi_count": 2, "koi_bin_oedp_sig": 0.6,
        "koi_steff": 5455, "koi_slogg": 4.5, "koi_smet": 0.0, "koi_srad": 0.93,
        "koi_smass": 0.92, "koi_kepmag": 15.3,
        "koi_dicco_msky": 0.1, "koi_dikco_msky": 0.12, "koi_fwm_stat_sig": 0.3,
    },
    "Likely false positive (eclipsing binary signature)": {
        "koi_period": 3.2, "koi_duration": 4.8, "koi_depth": 9000,
        "koi_impact": 0.9, "koi_ror": 0.15, "koi_dor": 8.0,
        "koi_prad": 18.0, "koi_teq": 1500, "koi_insol": 800, "koi_srho": 1.0,
        "koi_model_snr": 60.0, "koi_max_sngle_ev": 40.0, "koi_max_mult_ev": 55.0,
        "koi_num_transits": 80, "koi_count": 1, "koi_bin_oedp_sig": 0.05,
        "koi_steff": 6000, "koi_slogg": 4.2, "koi_smet": 0.1, "koi_srad": 1.1,
        "koi_smass": 1.05, "koi_kepmag": 13.8,
        "koi_dicco_msky": 5.4, "koi_dikco_msky": 5.1, "koi_fwm_stat_sig": 8.2,
    },
    "Ambiguous / weak-signal candidate": {
        "koi_period": 45.0, "koi_duration": 3.1, "koi_depth": 120,
        "koi_impact": 0.5, "koi_ror": 0.01, "koi_dor": 60.0,
        "koi_prad": 1.1, "koi_teq": 350, "koi_insol": 5.0, "koi_srho": 1.6,
        "koi_model_snr": 9.5, "koi_max_sngle_ev": 4.0, "koi_max_mult_ev": 8.0,
        "koi_num_transits": 6, "koi_count": 1, "koi_bin_oedp_sig": 0.4,
        "koi_steff": 5700, "koi_slogg": 4.4, "koi_smet": -0.1, "koi_srad": 1.0,
        "koi_smass": 1.0, "koi_kepmag": 14.9,
        "koi_dicco_msky": 0.8, "koi_dikco_msky": 0.9, "koi_fwm_stat_sig": 1.1,
    },
}

for name, feats in examples.items():
    out = predict_candidate(feats)
    print(f"=== {name} ===")
    print(json.dumps(out, indent=2))
    print()


## Section 12 — OPTIONAL: real raw light curve extension (needs internet)

Everything above runs on the KOI **catalog** table. If you also want to
show a *real* measured light curve (not simulated) going through actual
BLS period search and phase-folding - exactly Steps 1-3 of your original
pipeline diagram - run this section. It needs internet access to NASA's
MAST archive, so it won't work on a fully offline machine, and it's kept
separate so the rest of the notebook doesn't depend on it.

Set `RUN_THIS = True` below once you're ready.


In [ ]:
RUN_THIS = False   # flip to True to actually download and run this section

if RUN_THIS:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightkurve", "astropy"])

    import lightkurve as lk
    from astropy.timeseries import BoxLeastSquares

    kepid = 10797460   # Kepler-227, change to any KIC ID you like

    search = lk.search_lightcurve(f"KIC {kepid}", mission="Kepler", author="Kepler")
    lc = search.download_all().stitch()
    lc = lc.remove_nans().remove_outliers(sigma=5)
    flat_lc = lc.flatten(window_length=401)
    norm_lc = flat_lc.normalize()

    time = norm_lc.time.value
    flux = norm_lc.flux.value
    flux_err = norm_lc.flux_err.value if norm_lc.flux_err is not None else None

    bls = BoxLeastSquares(time, flux, dy=flux_err)
    period_grid = np.linspace(0.5, 50, 50000)
    bls_result = bls.power(period_grid, 0.1)

    best_idx = np.argmax(bls_result.power)
    best_period = bls_result.period[best_idx]
    best_t0 = bls_result.transit_time[best_idx]
    best_duration = bls_result.duration[best_idx]
    best_depth = bls_result.depth[best_idx]

    print(f"BLS best period   = {best_period:.4f} days")
    print(f"BLS transit depth = {best_depth*1e6:.1f} ppm")
    print(f"BLS duration      = {best_duration*24:.2f} hours")

    folded = norm_lc.fold(period=best_period, epoch_time=best_t0)
    global_view = folded.bin(time_bin_size=1/2001)

    plt.figure(figsize=(7, 4.5))
    plt.scatter(global_view.time.value, global_view.flux.value, s=4, color="#4a5568")
    plt.title(f"REAL phase-folded light curve - KIC {kepid} @ P={best_period:.3f}d")
    plt.xlabel("Phase"); plt.ylabel("Normalized flux")
    plt.tight_layout()
    plt.savefig(f"outputs/real_lightcurve_KIC{kepid}.png", dpi=150)
    plt.show()
else:
    print("Skipped - set RUN_THIS = True above to download and run on real data.")


## Wrap-up — what to screenshot for your PPT

- Section 1 plot → class distribution slide
- Section 2 → leakage explanation slide (text)
- Section 5/6/7 plots → results slide
- Section 8 plot → "we did this rigorously" slide (strongest credibility slide)
- Section 9 plot → "what a detection looks like" slide
- Section 11 printed output → live demo on stage
